In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn import model_selection

# Updated to point to the correct location of the competition data
DATA_PATH = "./data/champs-scalar-coupling"




In [2]:
train = pd.read_csv(
    f"{DATA_PATH}/train.csv",
    dtype={"id": np.int32, "atom_index_0": np.int16, "atom_index_1": np.int16},
)
test = pd.read_csv(
    f"{DATA_PATH}/test.csv",
    dtype={"id": np.int32, "atom_index_0": np.int16, "atom_index_1": np.int16},
)
sub = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")
print(train.shape, test.shape, sub.shape)

train["is_train"] = 1
test["is_train"] = 0
data = pd.concat([train, test], ignore_index=True)

structures = pd.read_csv(
    f"{DATA_PATH}/structures.csv",
    usecols=["molecule_name", "atom_index", "x", "y", "z"],
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        "x": np.float32,
        "y": np.float32,
        "z": np.float32,
    },
)

struct0 = structures.rename(
    columns={"atom_index": "atom_index_0", "x": "x0", "y": "y0", "z": "z0"}
)
struct1 = structures.rename(
    columns={"atom_index": "atom_index_1", "x": "x1", "y": "y1", "z": "z1"}
)
del structures

data = data.merge(struct0, how="left", on=["molecule_name", "atom_index_0"])
data = data.merge(struct1, how="left", on=["molecule_name", "atom_index_1"])
del struct0, struct1

dipole = pd.read_csv(
    f"{DATA_PATH}/dipole_moments.csv",
    dtype={
        "molecule_name": "category",
        "X": np.float32,
        "Y": np.float32,
        "Z": np.float32,
    },
)
dipole["dipole_mag"] = np.sqrt((dipole[["X", "Y", "Z"]] ** 2).sum(axis=1))
data = data.merge(
    dipole[["molecule_name", "dipole_mag"]], on="molecule_name", how="left"
)
del dipole

pot = pd.read_csv(
    f"{DATA_PATH}/potential_energy.csv",
    dtype={"molecule_name": "category", "potential_energy": np.float32},
)
data = data.merge(pot, on="molecule_name", how="left")
del pot

charges = pd.read_csv(
    f"{DATA_PATH}/mulliken_charges.csv",
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        "mulliken_charge": np.float32,
    },
)
charges0 = charges.rename(
    columns={"atom_index": "atom_index_0", "mulliken_charge": "charge0"}
)
charges1 = charges.rename(
    columns={"atom_index": "atom_index_1", "mulliken_charge": "charge1"}
)
data = data.merge(
    charges0[["molecule_name", "atom_index_0", "charge0"]],
    on=["molecule_name", "atom_index_0"],
    how="left",
)
data = data.merge(
    charges1[["molecule_name", "atom_index_1", "charge1"]],
    on=["molecule_name", "atom_index_1"],
    how="left",
)
del charges, charges0, charges1

data["charge_sum"] = data["charge0"] + data["charge1"]
data["charge_diff"] = data["charge0"] - data["charge1"]

contrib = pd.read_csv(
    f"{DATA_PATH}/scalar_coupling_contributions.csv",
    dtype={
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
        "fc": np.float32,
        "sd": np.float32,
        "pso": np.float32,
        "dso": np.float32,
    },
)
data = data.merge(
    contrib,
    on=["molecule_name", "atom_index_0", "atom_index_1", "type"],
    how="left",
)
del contrib

print(data.shape, sub.shape)

train = data[data.is_train == 1].drop(columns="is_train")
test = data[data.is_train == 0].drop(columns="is_train")
del data  # free memory




FileNotFoundError: [Errno 2] No such file or directory: './data/champs-scalar-coupling/train.csv'

In [3]:
train_p0 = train[["x0", "y0", "z0"]].fillna(0).values
train_p1 = train[["x1", "y1", "z1"]].fillna(0).values
test_p0 = test[["x0", "y0", "z0"]].fillna(0).values
test_p1 = test[["x1", "y1", "z1"]].fillna(0).values

train["dist"] = np.linalg.norm(train_p0 - train_p1, axis=1)
test["dist"] = np.linalg.norm(test_p0 - test_p1, axis=1)

train["dist_to_type_mean"] = train["dist"] / train.groupby("type")["dist"].transform(
    "mean"
)
test["dist_to_type_mean"] = test["dist"] / test.groupby("type")["dist"].transform(
    "mean"
)




NameError: name 'train' is not defined

In [4]:
train["type"] = train["type"].astype("category")
test["type"] = test["type"].astype("category")
train["type_code"] = train["type"].cat.codes.astype(np.int16)
test["type_code"] = test["type"].cat.codes.astype(np.int16)

col = [
    c
    for c in train.columns
    if c not in ["id", "molecule_name", "scalar_coupling_constant", "type"]
]

float_cols = [c for c in col if train[c].dtype.kind in "fc"]
train[float_cols] = train[float_cols].astype(np.float32)
test[float_cols] = test[float_cols].astype(np.float32)

type_counts = train["type"].value_counts()
train["sample_weight"] = 1.0 / train["type"].map(type_counts).astype(float)
test["sample_weight"] = 1.0  # placeholder; not used for prediction


def lgb_lmae_weighted(preds, dtrain):
    labels = dtrain.get_label()
    weight = dtrain.get_weight()
    mae = np.average(np.abs(labels - preds), weights=weight)
    return "lmae_weighted", np.log(mae + 1e-15), False  # lower is better


params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "none",
    "learning_rate": 0.02,
    "num_leaves": 384,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbosity": -1,
    "nthread": -1,
}




NameError: name 'train' is not defined

In [5]:
x_train, x_valid, y_train, y_valid, w_train, w_valid = model_selection.train_test_split(
    train[col],
    train["scalar_coupling_constant"],
    train["sample_weight"],
    test_size=0.2,
    random_state=99,
)

train_set = lgb.Dataset(x_train, label=y_train, weight=w_train)
valid_set = lgb.Dataset(x_valid, label=y_valid, weight=w_valid, reference=train_set)

model = lgb.train(
    params,
    train_set,
    num_boost_round=3000,
    valid_sets=[valid_set],
    feval=lgb_lmae_weighted,
    callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)],
)




NameError: name 'train' is not defined

In [6]:
test["scalar_coupling_constant"] = model.predict(
    test[col], num_iteration=model.best_iteration
)

test[["id", "scalar_coupling_constant"]].to_csv(
    "submission.csv", float_format="%.9f", index=False
)

NameError: name 'model' is not defined